# SciGuide Pipeline Simple Demo

Минимальный ноутбук для ручного контроля библиотеки:

- создаём `PipelineManager`;
- делаем `io.BytesIO` с текстом;
- прогоняем `chunking.run(...)`;
- прогоняем `search.run(...)`;
- смотрим, что ушло в библиотеку и что она вернула.

Перед запуском подними `qdrant` и `neo4j`, например:

```bash
docker compose -f notebooks/docker-compose.notebook.yml up -d
```

In [1]:
from __future__ import annotations

import io
import os
import sys
from pathlib import Path
from pprint import pprint
from urllib.parse import urlsplit, urlunsplit
from uuid import uuid4


def resolve_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent]
    for candidate in candidates:
        if (candidate / "src" / "packages").exists():
            return candidate
    raise RuntimeError("Run the notebook from the repository root or from notebooks/.")


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


ROOT_DIR = resolve_repo_root()
NOTEBOOK_DIR = ROOT_DIR / "notebooks"
load_env_file(NOTEBOOK_DIR / ".env")
load_env_file(ROOT_DIR / ".env")


def normalize_service_url(raw_url: str, *, fallback_host: str = "localhost") -> str:
    parts = urlsplit(raw_url)
    if parts.hostname not in {"qdrant", "neo4j", "db", "redis", "minio"}:
        return raw_url

    normalized_netloc = fallback_host
    if parts.port is not None:
        normalized_netloc = f"{fallback_host}:{parts.port}"
    if parts.username is not None:
        auth = parts.username
        if parts.password is not None:
            auth = f"{auth}:{parts.password}"
        normalized_netloc = f"{auth}@{normalized_netloc}"

    normalized_url = urlunsplit(
        (parts.scheme, normalized_netloc, parts.path, parts.query, parts.fragment)
    )
    print(f"Normalized service URL for local notebook: {raw_url} -> {normalized_url}")
    return normalized_url

PACKAGES_DIR = ROOT_DIR / "src" / "packages"
if str(PACKAGES_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGES_DIR))

from sciguide_pipeline import PipelineManager, SourceDocument

print(f"ROOT_DIR={ROOT_DIR}")
print(f"PACKAGES_DIR={PACKAGES_DIR}")

ROOT_DIR=/Users/timur/Desktop/FU/KR2
PACKAGES_DIR=/Users/timur/Desktop/FU/KR2/src/packages


In [2]:
QDRANT_URL = normalize_service_url(
    os.getenv("QDRANT_URL", "http://localhost:6333")
)
NEO4J_URI = normalize_service_url(
    os.getenv("PIPELINE_NEO4J_URI", "bolt://localhost:7687")
)
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
EMBEDDING_MODEL_NAME = os.getenv("PIPELINE_EMBEDDING_MODEL_NAME", "BAAI/bge-m3")
RERANKER_MODEL_NAME = os.getenv("PIPELINE_RERANKER_MODEL_NAME", "BAAI/bge-reranker-v2-m3")
MODEL_CACHE_DIR = str(ROOT_DIR / "hf_cache")


COLLECTION_NAME = f"sciguide_simple_demo_{uuid4().hex[:8]}"
GRAPH_NAMESPACE = COLLECTION_NAME

config_preview = {
    "qdrant_url": QDRANT_URL,
    "neo4j_uri": NEO4J_URI,
    "neo4j_username": NEO4J_USERNAME,
    "neo4j_database": NEO4J_DATABASE,
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "reranker_model_name": RERANKER_MODEL_NAME,
    "model_cache_dir": MODEL_CACHE_DIR,
    "collection_name": COLLECTION_NAME,
}
pprint(config_preview)

Normalized service URL for local notebook: http://qdrant:6333 -> http://localhost:6333
Normalized service URL for local notebook: bolt://neo4j:7687 -> bolt://localhost:7687
{'collection_name': 'sciguide_simple_demo_5abc2cf5',
 'embedding_model_name': 'BAAI/bge-m3',
 'model_cache_dir': '/Users/timur/Desktop/FU/KR2/hf_cache',
 'neo4j_database': 'neo4j',
 'neo4j_uri': 'bolt://localhost:7687',
 'neo4j_username': 'neo4j',
 'qdrant_url': 'http://localhost:6333',
 'reranker_model_name': 'BAAI/bge-reranker-v2-m3'}


In [3]:
sample_text = """
Graph-guided retrieval combines semantic search with a graph of extracted entities.
The pipeline first embeds document chunks and stores them in Qdrant.
Then the query is mapped to graph entities, expanded through neighboring nodes,
and the final ranking mixes vector similarity with graph score.
""".strip()

raw_stream = io.BytesIO(sample_text.encode("utf-8"))
raw_bytes = raw_stream.read()

document = SourceDocument(
    content=raw_bytes,
    document_id="simple-demo-doc-1",
    source_name="simple-demo.txt",
    metadata={
        "source": "simple notebook",
        "topic": "graph-guided retrieval",
    },
)

print("Bytes sent into SourceDocument:", len(raw_bytes))
print("Document object:")
pprint(document)

Bytes sent into SourceDocument: 296
Document object:
SourceDocument(content='Graph-guided retrieval combines semantic search with a '
                       'graph of extracted entities.\n'
                       'The pipeline first embeds document chunks and stores '
                       'them in Qdrant.\n'
                       'Then the query is mapped to graph entities, expanded '
                       'through neighboring nodes,\n'
                       'and the final ranking mixes vector similarity with '
                       'graph score.',
               metadata={'source': 'simple notebook',
                         'topic': 'graph-guided retrieval'},
               document_id='simple-demo-doc-1',
               source_name='simple-demo.txt')


In [4]:
with PipelineManager(
    qdrant_url=QDRANT_URL,
    qdrant_collection_name=COLLECTION_NAME,
    neo4j_uri=NEO4J_URI,
    neo4j_username=NEO4J_USERNAME,
    neo4j_password=NEO4J_PASSWORD,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    reranker_model_name=RERANKER_MODEL_NAME,
    model_cache_dir=MODEL_CACHE_DIR,
    graph_namespace=GRAPH_NAMESPACE,
    neo4j_database=NEO4J_DATABASE,
    search_limit=5,
    search_candidate_limit=20,
) as manager:
    chunk_report = manager.chunking.run([document])
    print("Chunking report:")
    pprint(chunk_report)

    search_report = manager.search.run(
        query="How does graph-guided retrieval mix vector search and graph score?",
        limit=5,
        candidate_limit=20,
    )

print("Search report:")
pprint(search_report)

/Users/timur/Desktop/FU/KR2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 48015.60it/s]


Chunking report:
ChunkingReport(documents_processed=1,
               chunks_created=1,
               collection_name='sciguide_simple_demo_5abc2cf5',
               graph_namespace='sciguide_simple_demo_5abc2cf5')
Search report:
SearchReport(query='How does graph-guided retrieval mix vector search and '
                   'graph score?',
             items=(SearchItem(chunk_id='simple-demo-doc-1:0',
                               document_id='simple-demo-doc-1',
                               text='Graph-guided retrieval combines semantic '
                                    'search with a graph of extracted '
                                    'entities.\n'
                                    'The pipeline first embeds document chunks '
                                    'and stores them in Qdrant.\n'
                                    'Then the query is mapped to graph '
                                    'entities, expanded through neighboring '
                              

In [5]:
print(f"Returned items: {len(search_report.items)}")
for index, item in enumerate(search_report.items, start=1):
    print(f"\n--- item #{index}")
    print("chunk_id:", item.chunk_id)
    print("document_id:", item.document_id)
    print("graph_only:", item.graph_only)
    print("vector_score:", item.vector_score)
    print("graph_score:", item.graph_score)
    print("final_score:", item.final_score)
    print("metadata:")
    pprint(item.metadata)
    print("text preview:")
    print(item.text[:500])

Returned items: 1

--- item #1
chunk_id: simple-demo-doc-1:0
document_id: simple-demo-doc-1
graph_only: False
vector_score: 0.7084145
graph_score: 21.281480240067975
final_score: 0.65
metadata:
{'source': 'simple notebook',
 'source_name': 'simple-demo.txt',
 'topic': 'graph-guided retrieval'}
text preview:
Graph-guided retrieval combines semantic search with a graph of extracted entities.
The pipeline first embeds document chunks and stores them in Qdrant.
Then the query is mapped to graph entities, expanded through neighboring nodes,
and the final ranking mixes vector similarity with graph score.
